In [24]:
all = [var for var in globals() if var[0] != "_"]
for var in all:
    del globals()[var]

del var, all
# This script clears all global variables except those starting with an underscore.

In [25]:
# Modeule imports
import numpy as np
import jax
import jax.numpy as jnp
import torch

import scipy.optimize as sci_opt
from scipy.optimize import NonlinearConstraint
import scipy.special
import module_opt_AD as my_opt

import matplotlib.pyplot as plt

import heat_equation_2D_for_students
import heat_equation_2D_for_students_jax

import importlib
importlib.reload(my_opt)
importlib.reload(heat_equation_2D_for_students)
importlib.reload(heat_equation_2D_for_students_jax)

from heat_equation_2D_for_students import HeatEquation2D
from heat_equation_2D_for_students_jax import HeatEquation2D_JAX

## Optimization - Finite Differentiation(FD) & Nonscaling

In [ ]:
lx = .04  # length in x direction
ly = .04  # length in y direction
z = .04  # height of the domain
N = 25    # number of grid points in x, y direction
k_Si = 149  # thermal conductivity of silicon
rho_Si = 2323  # density of silicon
cp_Si = 704.611  # specific heat capacity of silicon
CFL = .1 # CFL number

def T0(x:np.ndarray, y:np.ndarray) -> np.ndarray:
    """
    Initial temperature distribution over the domain.
    """
    T_0 = np.zeros_like(x)
    ## Cosine Case
    T_0 = 70 * np.sin(x * np.pi / lx) * np.sin(y * np.pi / ly) + 293
    return T_0

heq = HeatEquation2D(x=lx, y=ly, height=z, n_x=N, n_y=N, k=k_Si, rho=rho_Si, cp=cp_Si, CFL=CFL, init_condition=T0)

def q_gen(x:np.ndarray, y:np.ndarray, a:np.float64, b:np.float64, c:np.float64) -> np.ndarray:
    """
    Heat generation function over the domain.
    """
    q_g = a*x + b*y + c
    return q_g

In [ ]:
# ------------- Objective function -------------
w1 = .2
w2 = 1 - w1

def f_FD(x):
    # print(f'objective function starts to be evaluated')
    v = x[0]
    a = x[1]
    b = x[2]
    c = x[3]

    heq.reset()
    heq.set_fan_velocity(v=v)
    heq.set_heat_generation(q_gen, a=a, b=b, c=c)
    heq.solve_until_steady_state(tol=1e-3)
    # maxT = np.max(heq.u)
    beta = 10.0  # 로그섬지수
    maxT = scipy.special.logsumexp(beta * heq.u) / beta

    T_scaled = maxT

    # eta = -.002*v**2 + .08*v
    eta = heq.fan_efficiency

    return w1*T_scaled - w2*eta

# ------------- Constraint function -------------
def c_eq_FD(x):
    # print(f'constraint function starts to be evaluated')
    v, a, b, c = x[0], x[1], x[2], x[3]
    heq.set_heat_generation(q_gen, a=a, b=b, c=c)
    q_gen_total = heq.heat_generation_total

    return q_gen_total - 10.0  # Constraint: total heat generation must be 10.0 W

In [ ]:
iter_x_FD = []

iter_maxT_FD = []
iter_eta_FD = []

iter_obj_FD = []
iter_grad_obj_FD = []

iter_constr_FD = []
# iter_grad_constr = []

iter_grad_L_FD = []
# iter_grad_norm = []


def opt_iter_callback_FD(xk, state=None):
    """Called by trust-constr once per iteration with current xk."""
    print(f'\niteration callback starts !')
    v = xk[0]
    a = xk[1]
    b = xk[2]
    c = xk[3]
    fk = state.fun
    eta = -0.002 * v**2 + 0.08 * v
    maxT = 5 * (state.fun + 0.8 * eta) # since obj = w1*maxT - w2*eta
    constr = c_eq_FD(xk)*10

    iter_x_FD.append(np.array(xk, copy=True))
    iter_obj_FD.append(fk)
    iter_eta_FD.append(eta)
    iter_maxT_FD.append(maxT)  
    iter_constr_FD.append(constr)
    iter_grad_obj_FD.append(state.grad.copy())
    # iter_grad_constr.append([jac.copy() for jac in state.jac])
    iter_grad_L_FD.append(state.lagrangian_grad)

    print(f'Current x: {xk}')
    print(f'Current objective: {fk}')
    print(f'Current maxT: {maxT}')
    print(f'Current eta: {eta}')
    print(f'Current constraint value: {constr}')
    print(f'gradient of objective: {state.grad}')
    # print(f'gradient of constraints: {[jac for jac in state.jac]}')
    print(f'gradient of Lagrangian: {state.lagrangian_grad}')
    print()

In [ ]:
v0 = 5.0
a0 = 0.0
b0 = 0.0
c0 = (1.0*10**5 - 0.02*b0 - 0.02*a0)  # same formula as before
x0 = np.array([v0, a0, b0, c0])

# ---- Bounds for inputs ----
x_bounds = [
    (0.0, 30.0),        # v
    (-np.inf, np.inf),  # a
    (-np.inf, np.inf),  # b
    (0.0, np.inf),      # c
]

# Using NonlinearConstraint from scipy
nlc_FD = NonlinearConstraint(c_eq_FD, lb=0.0, ub=0.0)

# --- Optimization call with finite difference gradients ---
sci_opt_result = sci_opt.minimize(fun=f_FD, x0=x0, method='trust-constr', bounds=x_bounds, constraints=[nlc_FD], tol=1e-6, callback=opt_iter_callback_FD)

Iteration starts until steady state !
Steady State Reached in 1288 iterations with error 0.000997518835106348 at time 3.933351666698387 seconds.
Iteration starts until steady state !
Steady State Reached in 1288 iterations with error 0.0009975188941666602 at time 3.933351666698387 seconds.
Iteration starts until steady state !
Steady State Reached in 1288 iterations with error 0.000997518833742106 at time 3.933351666698387 seconds.
Iteration starts until steady state !
Steady State Reached in 1288 iterations with error 0.0009975188341400099 at time 3.933351666698387 seconds.
Iteration starts until steady state !
Steady State Reached in 1288 iterations with error 0.000997518697488431 at time 3.933351666698387 seconds.

iteration callback starts !
Current x: [np.float64(5.0), np.float64(0.0), np.float64(0.0), np.float64(100000.0)]
Current objective: -0.09601081144583357
Current maxT: 321.9810799697913
Current eta: 0.35000000000000003
Current constraint value: -3.477777777777776
gradient 